In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

folders = [f for f in dbutils.fs.ls("/Volumes/dataplatform01_central_dev_catalog_01/bronze_raw_sap_bo/sapbo_webi") if f.isDir()]
for folder in folders:
    if folder.name.startswith("prod/"):
        File_format = None
        for f in dbutils.fs.ls(folder.path):
            if f.isDir() and ("object_type=csv" in f.name or "object_type=pdf" in f.name ):
                File_format = folder.name.split("object_type=")[-1].split("/")[0]
                df = (
                    spark.read.format("binaryFile")
                    .option("recursiveFileLookup", "true")
                    .load(folder.path)
                    .selectExpr("_metadata.file_path as File_location")
                    .withColumns({
                        "Document_ID": F.regexp_extract(F.col("File_location"), r"WEBI_(.+?)\.(csv|pdf)", 1),
                        "File_format": F.regexp_extract(F.col("File_location"), r"\.([^./]+)$", 1)
                    })
                    .select("Document_ID", "File_format", "File_location")
                )
            df = df.withColumns({
                "format_priority": F.when(F.col("File_format") == "csv", 1).when(F.col("File_format") == "pdf", 2).otherwise(3),
                "ingestion_ts": F.try_to_timestamp(F.regexp_extract(F.col("File_location"), r"(\d{4}-\d{2}-\d{2}T\d{2}_\d{2}_\d{2})", 1), F.lit("yyyy-MM-dd'T'HH_mm_ss"))
            })

            window = (
                Window
                .partitionBy("Document_ID")
                .orderBy(F.col("format_priority"), F.col("ingestion_ts").desc_nulls_last())
            )

            df = (
                df.select("*", F.row_number().over(window).alias("row_num"))
                .filter(F.col("row_num") == 1)
                .drop("row_num", "format_priority", "ingestion_ts")
            )
        pass
summary = (
    df.groupBy("File_format")
    .count()
    .withColumnRenamed("count", "num_files")
)
display(summary)
cms_df = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms")
cms_window = (
    Window
    .partitionBy("Document_Id", "Data_Provider_ID", "SQL_Index")
    .orderBy(F.col("ingestion_ts").desc_nulls_last())
)

cms_df = (
    cms_df
    .withColumn("row_num", F.row_number().over(cms_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)
cms_df = cms_df.filter(F.col("Document_Id").isin(
    [row.Document_ID for row in df.filter(F.col("File_location").isNull()).select("Document_ID").distinct().collect()]
))
joined_df = (
    cms_df.drop("File_format", "File_location")
    .join(df.select("Document_ID", "File_format", "File_location"), cms_df.Document_Id == df.Document_ID, "left")
    .drop(df.Document_ID)
)
print(f"Rows to write back to Bronze cms table:{joined_df.count()}")
joined_df = joined_df.withColumn("ingestion_ts", F.current_timestamp())
joined_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms")